# GR00T + Cosmos-Predict2 — Inference-Only Playground (Google Colab)

This notebook runs **inference only** (no training/fine-tuning) with:
- **NVIDIA Isaac GR00T N1.5-3B** — the vision-language-action policy model
- **NVIDIA Cosmos-Predict2 (2B Video2World)** — the small world/prediction model

### GPU requirement (important)
GR00T N1.5 needs `flash-attn`, which only runs on **Ampere-or-newer** NVIDIA GPUs (A100, L4, RTX 30/40-series, H100).
Free Colab gives you a **T4 (Turing)** GPU, which flash-attn does **not** support.

- If you're on **Colab Pro/Pro+**: `Runtime > Change runtime type > A100` or `L4` — run everything below.
- If you're on **free Colab (T4)**: the GPU-check cell will detect this and skip the GR00T cells automatically — you can still run the Cosmos-Predict2 section, which does not require flash-attn.

Run cells top to bottom.

## 0. Check your GPU

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type > GPU"
cc_major, cc_minor = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
SUPPORTS_FLASH_ATTN = cc_major >= 8  # Ampere = 8.x, Ada = 8.9, Hopper = 9.x
print(f"\nDetected GPU: {GPU_NAME} (compute capability {cc_major}.{cc_minor})")
print(f"Flash-attention 2 compatible: {SUPPORTS_FLASH_ATTN}")
if not SUPPORTS_FLASH_ATTN:
    print("\n>>> This is a T4 or older GPU. The GR00T cells below will be skipped.")
    print(">>> You can still run the Cosmos-Predict2 section further down.")
    print(">>> To run GR00T, switch to a Colab Pro A100/L4 runtime.")

---
## Part A — GR00T N1.5-3B inference (requires A100/L4/Ampere+)

This section installs the GR00T repo, downloads the pretrained checkpoint (~6GB, auto from Hugging Face),
and runs a single forward pass on a **dummy observation** — no robot, no sim, no training.

Goal: see the raw input/output contract of the model (what an image+instruction turns into as an action chunk).

In [ ]:
if SUPPORTS_FLASH_ATTN:
    !git clone --recurse-submodules https://github.com/NVIDIA/Isaac-GR00T.git
    %cd Isaac-GR00T
    !pip install -e .[base] -q
    !pip install --no-build-isolation flash-attn -q
else:
    print("Skipping GR00T install — GPU does not support flash-attn.")

In [ ]:
if SUPPORTS_FLASH_ATTN:
    import numpy as np
    from gr00t.model.policy import Gr00tPolicy
    from gr00t.data.embodiment_tags import EmbodimentTag

    # Loads nvidia/GR00T-N1.5-3B automatically from Hugging Face on first run
    policy = Gr00tPolicy(
        model_path="nvidia/GR00T-N1.5-3B",
        embodiment_tag=EmbodimentTag.NEW_EMBODIMENT,
        device="cuda",
    )

    # Build a fake observation: one camera frame + a language instruction + dummy proprioception
    fake_image = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    observation = {
        "video.image_0": fake_image[None],           # add batch/time dim as required
        "state.arm": np.zeros((1, 7), dtype=np.float32),
        "annotation.human.action.task_description": ["pick up the red block"],
    }

    action_chunk = policy.get_action(observation)
    print("Action chunk keys:", list(action_chunk.keys()))
    for k, v in action_chunk.items():
        print(f"  {k}: shape={np.array(v).shape}")
else:
    print("Skipped — no flash-attn support on this runtime.")

**Note:** the exact key names for `observation` (e.g. `video.image_0`, `state.arm`) depend on the
`modality.json` / data-config for the embodiment tag you use. Using `EmbodimentTag.NEW_EMBODIMENT`
with dummy shapes is just to see the mechanics; for anything real you'd follow GR00T's
`LeRobot`-compatible data schema (see `getting_started/` in the repo).

---
## Part B — Cosmos-Predict2 (2B Video2World) inference

Runs on T4 or better. Given a starting frame + text description of an action,
it generates a short predicted future clip — the "mental rehearsal" piece.

First run requires a Hugging Face token (Cosmos checkpoints are gated) — create one at
https://huggingface.co/settings/tokens and paste it when prompted.

In [ ]:
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" transformers accelerate imageio imageio-ffmpeg

from huggingface_hub import login
login()  # paste your HF token when prompted (needs access to nvidia/Cosmos-Predict2)

In [ ]:
import torch
from diffusers import Cosmos2VideoToWorldPipeline
from diffusers.utils import load_image, export_to_video

model_id = "nvidia/Cosmos-Predict2-2B-Video2World"

pipe = Cosmos2VideoToWorldPipeline.from_pretrained(model_id, torch_dtype=torch.bfloat16)
pipe.to("cuda")

# Starting frame: swap this URL for any image relevant to your robot scene
start_image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cosmos/cosmos-video2world-input.jpg"
)

prompt = "A robotic arm reaches toward a small red block on a table and picks it up smoothly."

output = pipe(
    image=start_image,
    prompt=prompt,
    num_frames=25,
    guidance_scale=7.0,
).frames[0]

export_to_video(output, "predicted_future.mp4", fps=8)
print("Saved predicted_future.mp4 — download it from the Colab file browser (left sidebar).")

---
## Part C — Toy "propose then imagine" loop

This stitches A and B together conceptually: GR00T proposes an action/instruction,
Cosmos-Predict2 imagines the resulting future frame. On free T4, only the Cosmos
half will actually run — the GR00T "proposal" step is simulated with a hardcoded
instruction so you can still see the shape of the pipeline.

In [ ]:
# Step 1: get a proposed action/instruction
if SUPPORTS_FLASH_ATTN:
    proposed_instruction = "pick up the red block"  # in a real setup: derive this from action_chunk
else:
    proposed_instruction = "pick up the red block"  # hardcoded stand-in since GR00T was skipped
    print("(GR00T skipped on this GPU — using a hardcoded instruction as a stand-in.)")

# Step 2: ask Cosmos-Predict2 to imagine the consequence
output = pipe(
    image=start_image,
    prompt=f"A robotic arm executes the instruction: {proposed_instruction}.",
    num_frames=25,
    guidance_scale=7.0,
).frames[0]

export_to_video(output, "proposed_then_imagined.mp4", fps=8)
print("Saved proposed_then_imagined.mp4")

---
### Where to go from here
- Swap the dummy GR00T observation for a real image from your own webcam/dataset.
- Swap `NEW_EMBODIMENT` + dummy shapes for a real `modality.json` config once you have real robot data.
- For your thesis, this loop (propose → imagine → evaluate) is the toy version of the Skill Transformer + Cosmos world-model interaction — the real version would use the predicted frame to score/select among multiple candidate skill tokens rather than just visualizing one.